# S&P 500 Options: LSTM

This notebook fits the declared LSTM member of the sequence population snapshotted by
`09_deep_learning`. Chronological windows, validation gaps, checkpoints, and prediction
eligibility are resolved through the shared sequence boundary.

Prerequisite: `09_deep_learning` must create the complete official sequence population.

**Why the population is declared in one notebook and filled by several.** The set of members is
a claim made once, before any of them is fitted, so that no family can be added or dropped after
its results are visible. This notebook fits one declared member into a population it did not
define and cannot extend; running it alone leaves the population incomplete rather than smaller.

## What this model is, and what it is being asked to do here

An LSTM reads a symbol's history one session at a time and carries a state forward, updating it
at each step through gates that decide how much of the new observation to admit and how much of
the existing state to keep. The gates are what separate it from a plain recurrent network: they
give the model a route by which information from many steps back can reach the output without
being multiplied away at every step, which is what makes a long lookback usable at all.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# S&P 500 選擇權：LSTM

本 Notebook 擬合 `09_deep_learning` 所建立快照中，已宣告的 LSTM 序列模型成員。時間順序視窗、驗證間隔、檢查點與預測資格，均透過共用序列模型邊界解析。

先備條件：`09_deep_learning` 必須先建立完整的正式序列模型群。

**為何在一份 Notebook 宣告模型群，卻由多份 Notebook 填入。** 成員集合只在擬合任何成員前宣告一次，使任何模型家族都無法在看見結果後新增或移除。本 Notebook 將一個已宣告成員擬合到自己未定義、也無法擴充的模型群；若單獨執行，模型群會維持未完成狀態，而非縮小成員集合。

## 模型是什麼，以及此處要求它做什麼

LSTM 每次讀取某檔股票歷史資料的一個交易日，將狀態向前傳遞，並在每一步透過閘門更新；閘門決定接納多少新觀測資料，以及保留多少既有狀態。這些閘門是它與一般循環網路的差異：它們讓多步以前的資訊有路徑傳至輸出，不會在每一步相乘後逐漸消失，因此較長回看期間才有可能使用。
<!-- ML4T_ZH_END c0000-p000 -->

**What that buys on this data, and what it costs.** The cross-sectional families in this case
study see one row per symbol per decision time: whatever history matters has to have been
compressed into a feature first. This model is handed the window instead and left to decide what
in it matters, so a pattern nobody wrote a feature for is reachable. The cost is that it has far
more freedom to fit noise, and options data on a few hundred names is not abundant, so the
comparison against the cross-sectional families is the point of running it rather than a
formality.

**It is not expected to win, and that is worth saying before the numbers.** A sequence model
earns its keep where the ordering of observations carries information the features do not. If
it does not beat a gradient-boosted model on engineered features here, that is a result about
this data, not a failed run, and the chapter reports it either way.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**在此資料上能得到什麼，代價又是什麼。** 本案例研究中的橫斷面模型家族，在每個決策時點對每檔股票只看到一列：任何重要歷史資訊，都必須先壓縮成特徵。本模型則直接收到整個視窗，自行判斷其中什麼重要，因此有機會發現無人預先寫成特徵的型態。代價是它有更大自由度擬合雜訊，而幾百檔股票的選擇權資料並不充裕；因此，與橫斷面模型家族比較，是執行它的主要目的，而非例行程序。

**事前並不預期它會勝出，這點值得在看到數字前說明。** 序列模型的價值，來自觀測順序所含、既有特徵未捕捉的資訊。若此處未能超越以人工設計特徵訓練的梯度提升模型，那是關於資料的結果，而非執行失敗；章節會如實報告。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""Fit the declared S&P 500 options LSTM request."""

import polars as pl

from case_studies.sp500_options.research_workflow import (
    ALL_LABELS,
    declared_dl_device,
    model_request_catalog,
    open_study,
    published_dl_device,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_subset,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
DEVICE: str = ""

POPULATION_NAME: str = ""

### The device the population was fitted on

A network trained on a GPU and the same network trained on a CPU accumulate their sums in a
different order and reach different weights, so the device is part of what the fitted model is
and sits inside the training identity rather than beside it. The device this population was
fitted on is declared once, in `modeling.dl.device` in `config/setup.yaml`, and read from there
by all four deep-learning notebooks rather than retyped in each. On a machine with no NVIDIA
card the run stops here rather than quietly training something else: set `DEVICE="cpu"` and pass
a `POPULATION_NAME` to fit the same requests there, under a name of their own.

**Why a second name rather than a second run under the first.** The published population is a
claim about a specific set of fitted models. A CPU fit of the same request is a different set,
close but not identical, and letting it join the published name would make the population mean
"these requests, fitted somewhere" instead of "these models". The check above refuses that
combination outright rather than warning about it, because a warning in a long run is read once
and then not read.

**This is why the gradient-boosted families run on CPU and these run on GPU.** A reader without
a card can reproduce everything the book compares on trees; the sequence families are the part
that needs hardware, and they are separated so that the absence of a GPU costs a chapter's
comparison rather than the whole case study.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
### 模型群擬合時使用的裝置

相同網路分別在 GPU 與 CPU 上訓練，加總順序不同，最終權重也可能不同，因此運算裝置屬於已擬合模型定義的一部分，應記入訓練身分，而非只放在旁邊。本模型群使用的裝置，透過 `modeling.dl.device` 於 `config/setup.yaml` 中宣告一次，由四份深度學習 Notebook 共同讀取，不各自重填。若電腦沒有 NVIDIA 顯示卡，執行會在此停止，而非悄悄訓練不同模型；可設定 `DEVICE="cpu"` 並傳入 `POPULATION_NAME`，在該裝置上以自身名稱擬合相同請求。

**為何須用另一個名稱，而非在原名稱下再執行一次。** 已發布模型群宣告的是特定一組已擬合模型。即使請求相同，CPU 擬合結果也是另一個集合，與原本結果接近但不完全相同。若允許其加入已發布名稱，模型群意義便會從「這些模型」變成「這些請求在某處擬合的結果」。上方檢查會直接拒絕這種混用，而非只警告，因為漫長執行中的警告通常只被看過一次，之後就被忽略。

**這也是梯度提升模型家族在 CPU 上執行、序列模型家族在 GPU 上執行的原因。** 沒有顯示卡的讀者仍可重現書中所有樹模型比較；序列模型家族才需要特定硬體。兩者分開，缺少 GPU 只會影響一章的比較，而非整個案例研究。
<!-- ML4T_ZH_END c0003-p000 -->


In [3]:
CANONICAL_POPULATION_NAME = "sp500-options-sequence-validation-v1"

published_device = published_dl_device()
device = declared_dl_device(DEVICE)
population_name = POPULATION_NAME or CANONICAL_POPULATION_NAME
if device != published_device and population_name == CANONICAL_POPULATION_NAME:
    raise ValueError(
        f"this run fits on {device!r}, not the published {published_device!r}, so its "
        f"identities are not the ones {CANONICAL_POPULATION_NAME!r} holds; pass "
        f"POPULATION_NAME to give them a population of their own"
    )
print(f"training device: {device} (declared: {published_device})")

training device: cuda (declared: cuda)


## Declared request

**What the settings decide.** `lookback: 60` is the window handed to the model: sixty sessions,
about a quarter, so a fitted state can span an earnings cycle without reaching back to a regime
the symbol has left. `hidden_size: 64` and `n_layers: 2` set how much the state can hold and how
many times it is re-read before the output; larger values fit more and generalize less, and on a
panel this size they are the first place overfitting shows. `dropout: 0.1` drops a tenth of the
connections on each training pass, which stops the network leaning on any single one.

`batch_size: 2048` is a throughput choice rather than a modelling one, but it is not neutral:
gradient noise falls as the batch grows, so a large batch trains more smoothly and explores
less. It is declared rather than tuned because tuning it would change what was fitted while
looking like an infrastructure decision.

**The configuration is read from a preset, not written here.** `lstm_h64` names a file under
`case_studies/config/`, so this notebook cannot quietly differ from the same architecture in
another chapter, and a reader comparing the two is comparing declarations rather than code.

**Every label is fitted, not just the primary one.** The request spans `ALL_LABELS`, because
selection downstream ranks across labels as well as across configurations, and a label with no
candidates cannot be chosen or ruled out.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 已宣告的請求

**各設定決定什麼。** `lookback: 60` 是交給模型的視窗：六十個交易日，約一季，讓已擬合狀態能涵蓋一次財報週期，而不回溯至股票已離開的市場狀態。`hidden_size: 64` 與 `n_layers: 2` 設定狀態容量，以及輸出前重新讀取狀態的次數；數值越大，模型能擬合越多、泛化能力越弱。在此規模的資料面板上，過度擬合最早會在這裡顯現。`dropout: 0.1` 在每次訓練時丟棄十分之一的連結，避免網路過度依賴某一條。

`batch_size: 2048` 是吞吐量選擇，而非直接的建模選擇，但並非中性：批次越大，梯度雜訊越小，訓練較平滑，探索也較少。此值預先宣告、不做調整，因為調整它會改變擬合內容，卻可能被誤認為只是基礎設施決策。

**設定來自預設檔，而非寫死於此。** `lstm_h64` 指定 `case_studies/config/` 下的檔案，避免此 Notebook 與其他章節中的相同架構悄悄不同；讀者比較兩者時，便可比較宣告，而非程式碼。

**會擬合所有標籤，不只有主要標籤。** 請求涵蓋 `ALL_LABELS`，因為下游會跨標籤與設定一起排名；若某標籤沒有候選模型，就無法選用，也無法排除。
<!-- ML4T_ZH_END c0005-p000 -->


In [4]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
requests = model_request_catalog(
    "deep_learning",
    labels=ALL_LABELS,
    config_names=("lstm_h64",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""regression""",54,273,85083,2,2019-01-07 00:00:00,2020-11-25 00:00:00,20,"""canonical""","""9be44756048d"""


## Execute and validate

The shared sequence runner owns chronological window construction, fold fitting, fitted-state
reload, checkpoint publication, restart, and exact eligible-key validation.

**A checkpoint is part of a configuration, not a detail of how it was fitted.** Training runs for
100 epochs and publishes every fifth, so this one request becomes twenty scored candidates rather
than one. That is deliberate: a network's validation performance is not monotone in training
time, and the epoch at which it peaks is a property of the fit that a reader is entitled to see
rather than a number chosen after the fact. Each published checkpoint therefore carries its own
identity and competes on its own downstream, and picking the best epoch after seeing the results
is selection, which happens once, downstream, on backtests.

**Restart is a correctness property, not a convenience.** Fold fits are written as they finish
and reloaded rather than refitted, so a run interrupted after eight of ten folds resumes at the
ninth. What matters is not the time saved: it is that the alternative - starting over - invites
quietly reducing the job to make it fit, and a population assembled from a reduced re-run and a
full first attempt is not one population. Reloading a fitted state means the checkpoint that
reaches the registry is the one the schedule asked for, whatever happened to the process.


<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 執行與驗證

共用序列模型執行器負責依時間順序建構視窗、資料折擬合、重新載入已擬合狀態、發布檢查點、重新啟動，以及精確驗證符合條件的鍵值。

**檢查點是設定的一部分，而非擬合過程的附帶細節。** 訓練共進行 100 個 epoch，每五個發布一個檢查點，因此單一請求會變成二十個受評分候選模型，而非一個。這是刻意安排：神經網路的驗證表現不會隨訓練時間單調變化；達到高峰的 epoch 是擬合結果本身的性質，讀者應能看見，而非事後挑選的數字。因此每個已發布檢查點都有自身身分，在下游獨立競爭；看過結果後挑選最佳 epoch 屬於選擇，只在下游依回測結果進行一次。

**重新啟動攸關正確性，不只是方便。** 各資料折完成擬合時即寫出，之後重新載入而非重擬合。因此若十折中的前八折完成後執行中斷，可從第九折繼續。關鍵不只是節省時間：若替代做法是從頭開始，容易誘使人悄悄縮減工作量以求完成；若模型群混合縮減後重跑與第一次完整嘗試的結果，就不是同一模型群。重新載入已擬合狀態，使最後進入登錄系統的檢查點，無論程序中途發生什麼，都符合排程要求。
<!-- ML4T_ZH_END c0007-p000 -->

**Windows are built chronologically and never span a fold boundary.** A sequence handed to the
model has to end before the fold's validation window opens, or the state carries information
from the period being scored. The runner owns that construction for the same reason the fold
geometry is shared: it is the kind of rule that is easy to restate slightly differently in each
notebook and impossible to notice when someone does.

<!-- ML4T_ZH_BEGIN c0007-p001 -->
**視窗依時間順序建立，絕不跨越資料折邊界。** 交給模型的序列必須在資料折驗證視窗開始前結束，否則狀態會帶入受評分期間的資訊。執行器統一負責建構，理由與共用資料折結構相同：若各 Notebook 分別重述規則，容易出現細微差異，也很難察覺。
<!-- ML4T_ZH_END c0007-p001 -->


In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_subset(
        study,
        resolved,
        population=population_name,
    )
else:
    if not WORKSPACE or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Fold-major CV: 2 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=80,175 seq across 532 symbols
    val=56,536 seq across 556 symbols
    creating datasets...
    datasets ready
    lstm_h64:


      epoch   1/100: train_loss=0.665532


      epoch   2/100: train_loss=0.643745


      epoch   3/100: train_loss=0.606749


      epoch   4/100: train_loss=0.569481


... (198 outputs left out)

      epoch 100/100: train_loss=0.111032, val_loss=2.794389, IC=+0.0508


      best_ep=35, IC=+0.0639 (463.4s, 20 checkpoints)


  lstm_h64: best_epoch=45, IC=+0.0265 (844.3s)



  Best: lstm_h64 @ epoch 45 (IC=+0.0265)
  Saved to case_studies/sp500_options/run_log/training/9be44756048d/diagnostics


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("LSTM execution returned a partial checkpoint")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",5,"""canonical""",true,"""9be44756048d""","""0f548a2109cf"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",10,"""canonical""",true,"""9be44756048d""","""f842b4cd929f"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",15,"""canonical""",true,"""9be44756048d""","""d92ff679e5ab"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",20,"""canonical""",true,"""9be44756048d""","""cdd4268a446a"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",25,"""canonical""",true,"""9be44756048d""","""eb6405cd502c"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",80,"""canonical""",true,"""9be44756048d""","""01e68c8df7ed"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",85,"""canonical""",true,"""9be44756048d""","""43321039ac1e"""
"""deep_learning""","""ret_to_expiry""","""lstm_h64""","""epoch""",90,"""canonical""",true,"""9be44756048d""","""3d8250327e5f"""


The complete LSTM checkpoint population is ready for model analysis and backtesting. This
notebook does not compare it with another family or choose a checkpoint.

**What completeness means here and why it is checked before anything leaves.** Every requested
checkpoint produced predictions on exactly the rows its eligibility contract declared - not
more, and not fewer. A partial checkpoint is refused rather than published, because a downstream
comparison against a model scored on a subset of the panel is not a comparison, and the subset
is invisible by the time anyone reads the result.

**The eligible rows are fewer than the cross-sectional families see, and that is structural.**
A symbol cannot be scored until sixty sessions of it exist, so this family is eligible on
strictly fewer rows than a model reading one row at a time. `11_model_analysis` groups by
eligibility for exactly this reason: comparing an IC from this population against one from a
cross-sectional population mixes the models with the rows they were scored on.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
完整 LSTM 檢查點模型群已準備好進行模型分析與回測。本 Notebook 不與其他模型家族比較，也不選擇檢查點。

**此處的完整性代表什麼，以及為何在任何內容輸出前檢查。** 每個要求的檢查點，恰好在資格規則宣告的資料列上產生預測，不多也不少。部分完成的檢查點會被拒絕，而非發布，因為若下游拿只在資料面板子集上評分的模型來比較，就不是有效比較；而讀取結果時，子集通常已不可見。

**符合條件的資料列少於橫斷面模型家族，這是結構性差異。** 某檔股票未累積六十個交易日前，不能評分；因此此模型家族可用資料列必然少於一次只讀一列的模型。`11_model_analysis` 正因如此依資格分組：若直接將此模型群的資訊係數與橫斷面模型群比較，模型差異就會與受評分資料列差異混在一起。
<!-- ML4T_ZH_END c0010-p000 -->
